In [ ]:
import pandas as pd
import os
import argparse
import numpy as np
import pymc as pm
import arviz as az
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import ajr_causalpy_iv as ajr
import pytensor
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import xarray as xr

%reload_ext autoreload
%autoreload 2


In [ ]:
# ---- Priors (aligned to standardized design) ----
# instruments (beta_t dims): [Intercept, logem4_std]
# structural (beta_z dims):  [Intercept, avexpr_std, logem4_std]  (only if loosen_exclusion=True)
priors_delta = {
    "mus": [
        [0.0, 0.0],                 # beta_t means: Intercept, logem4_std
        [0.0, 0.0, -0.2946],         # beta_z means: Intercept, avexpr_std, logem4_std
    ],
    "sigmas": [
        [1.0, 1.0],                 # beta_t sds
        [1.0, 1.0, 1.0],           # beta_z sds (tight prior on logem4_std if loosened)
    ],
    "eta": 2,       # LKJ shape (correlations); η=2 = mild shrinkage toward 0A
    "lkj_sd": 2,    # rate for Exponential prior on residual SDs (if you kept Exponential)
}

data_path = r"C:\Users\B375471\Downloads\Acemoglu_osv\colonial_origins\maketable4\maketable4.dta"

# New loader returns (df, stats); we only need df here (run() standardizes internally)
df_ajr, stats = ajr.load_ajr_data(data_path, baseline_only=True, standardize=True)

# ---- Fit model ----
iv_obj, idata = ajr.run(
    data=df_ajr,              
    priors=priors_delta,
    draws=2000,
    tune=1000,
    chains=4,
    cores=4,
    target_accept=0.9,
    loosen_exclusion=True,    
    covariates=None,          
    netcdf_path="ajr_iv_delta_posterior00.nc",
    standardize=True, 
    stats=stats        
)




In [ ]:
# ---- Priors (aligned to standardized design) ----
# instruments (beta_t dims): [Intercept, logem4_std]
# structural (beta_z dims):  [Intercept, avexpr_std, logem4_std]  (only if loosen_exclusion=True)
priors_delta = {
    "mus": [
        [0.0, 0.0],                 # beta_t means: Intercept, logem4_std
        [0.0, 0.0, 0.0],         # beta_z means: Intercept, avexpr_std, logem4_std
    ],
    "sigmas": [
        [1.0, 1.0],                 # beta_t sds
        [1.0, 1.0, 1.0],           # beta_z sds (tight prior on logem4_std if loosened)
    ],
    "eta": 2,       # LKJ shape (correlations); η=2 = mild shrinkage toward 0A
    "lkj_sd": 2,    # rate for Exponential prior on residual SDs (if you kept Exponential)
}

data_path = r"C:\Users\B375471\Downloads\Acemoglu_osv\colonial_origins\maketable4\maketable4.dta"

# New loader returns (df, stats); we only need df here (run() standardizes internally)
df_ajr, stats = ajr.load_ajr_data(data_path, baseline_only=True, standardize=True)

# ---- Fit model ----
iv_obj, idata = ajr.run(
    data=df_ajr,              
    priors=priors_delta,
    draws=2000,
    tune=1000,
    chains=4,
    cores=4,
    target_accept=0.9,
    loosen_exclusion=True,    
    covariates=None,          
    netcdf_path="ajr_iv_delta_posterior03.nc",
    standardize=True, 
    stats=stats        
)

In [ ]:
mu_delta_std = -0.236 * stats["logem4"]["std"] 
print(f"Prior mean for delta (standardized): {mu_delta_std:.4f}")
mu_delta_sd_std = 0.1799 * stats["logem4"]["std"]
print(f"Prior sd for delta (standardized): {mu_delta_sd_std:.4f}")

mu_beta_std = 0.38 * stats["avexpr"]["std"]
print(f"Prior mean for beta (standardized): {mu_beta_std:.4f}")

In [ ]:
idata2 = az.from_netcdf("ajr_iv_delta_posterior00.nc")
az.plot_trace(idata2, var_names=["beta_z", "beta_t"])
az.plot_posterior(idata2, var_names=["beta_orig", "delta_orig", "intercept_orig"], hdi_prob=0.95)


In [ ]:


axes = az.plot_posterior(
    idata2,
    var_names=["beta_orig", "delta_orig"],
    hdi_prob=0.95,
    figsize=(10, 4),
    labeller=az.labels.MapLabeller(
        var_name_map={
            "beta_orig": r"$\beta$ (Protection against expropriation)",
            "delta_orig": r"$\delta$ (Settler mortality)"
        }
    )
)

# Extract the figure from one of the axes
fig = axes[0].get_figure()

# Add a shared title
fig.suptitle("Structural Parameters", fontsize=14, y=1.05)

plt.tight_layout()
plt.show()


In [ ]:


# Flatten posterior samples
beta = idata2.posterior["beta_orig"].values.flatten()
delta = idata2.posterior["delta_orig"].values.flatten()

# ---- Aesthetic setup ----
az.style.use("seaborn-v0_8-darkgrid")
plt.figure(figsize=(7, 6))

# ---- KDE contour ----
ax = sns.kdeplot(
    x=delta,
    y=beta,
    fill=True,
    levels=[0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95],
    cmap="mako",         # or "mako", "crest", "rocket", "cividis"
    linewidths=0.5
)

# ---- Labels and title ----
ax.set_xlabel(r"$\delta$ (Settler mortality)", fontsize=13, labelpad=10)
ax.set_ylabel(r"$\beta$ (Protection against expropriation)", fontsize=13, labelpad=10)
ax.set_title(r"Joint Posterior of $\beta$ and $\delta$", fontsize=15, pad=15)

# ---- Reference lines ----
ax.axvline(0, color="black", ls="--", lw=1)
ax.axhline(0, color="black", ls="--", lw=1)

# ---- Margins, ticks, and style ----
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", which="major", labelsize=11)
ax.grid(alpha=0.2)

# ---- Optional: mean point marker ----
ax.scatter(delta.mean(), beta.mean(), color="red", s=40, marker="x", label="Posterior mean")
ax.legend(frameon=False, loc="upper right", fontsize=11)
ax.set_xlim(-1.0, 0.4)
ax.set_ylim(-0.8, 1.5)


plt.show()



In [ ]:
az.plot_trace(
    idata2,
    var_names=["beta_z", "beta_t"],
    labeller=az.labels.MapLabeller(
        var_name_map={"beta_z": "Structural parameters", "beta_t": "First stage parameters"}
    )
)



idata2_plot = idata2.copy()

# Find which dimension indexes beta_z (often "beta_z_dim_0" or similar)
dim_name = idata2_plot.posterior["beta_z"].dims[-1]

# Replace the coordinate label
idata2_plot.posterior = idata2_plot.posterior.assign_coords({
    dim_name: [
        "Intercept" if x == "Intercept" else
        "Protection against expropriation" if x == "avexpr" else
        "Settler mortality" if x == "logem4" else x
        for x in idata2_plot.posterior[dim_name].values
    ]
})

coef_dim = idata2_plot.posterior["beta_z"].dims[-1]
idata_no_intercept = idata2_plot.copy()
idata_no_intercept.posterior = idata_no_intercept.posterior.drop_sel({coef_dim: "Intercept"})

az.plot_posterior(
    idata_no_intercept,
    var_names=["beta_z"],
    hdi_prob=0.95,
    labeller=az.labels.MapLabeller(var_name_map={"beta_z": "Structural parameters"})
)

In [ ]:


posterior = idata2.posterior
y_obs = df_ajr["logpgp95"].to_numpy()
z_inst = df_ajr["logem4"].to_numpy()

# ---------- Build structural predictions: yhat(chain, draw, obs) ----------
def find_obs_dim(da):
    for cand in ("obs", "observation", "i", "n", "N", "obs_id"):
        if cand in da.dims:
            return cand
    return None

beta_z = posterior["beta_z"]
obs_dim = find_obs_dim(beta_z)

if obs_dim is not None:
    structural_preds = beta_z  # already (chain, draw, obs)
else:
    if "covariates" not in beta_z.dims:
        raise ValueError(f"'beta_z' needs obs dim or 'covariates'. Got {beta_z.dims}.")
    covs_all = [str(c) for c in beta_z.coords["covariates"].values]
    intercept_names = {"Intercept", "intercept", "const", "Constant", "_cons"}
    use_explicit_intercept = "intercept_orig" in posterior.data_vars
    covs = [c for c in covs_all if not (use_explicit_intercept and c in intercept_names)]

    n_obs = len(df_ajr)
    cols = []
    for c in covs:
        if c in intercept_names:
            cols.append(np.ones((n_obs, 1)))  # only if no explicit intercept param
        else:
            if c not in df_ajr.columns:
                raise KeyError(f"Missing covariate in df_ajr: {c!r}")
            cols.append(df_ajr[c].to_numpy().reshape(-1, 1))
    X = np.hstack(cols) if cols else np.zeros((n_obs, 0))

    B = beta_z.sel(covariates=covs).values                      # (chain, draw, K)
    yhat = np.einsum("cdk,nk->cdn", B, X) if X.shape[1] else np.zeros(B.shape[:2] + (n_obs,))
    if use_explicit_intercept:
        yhat = yhat + posterior["intercept_orig"].values[..., None]

    structural_preds = xr.DataArray(
        yhat, dims=("chain","draw","obs"),
        coords={
            "chain": posterior.get("chain", xr.DataArray(np.arange(yhat.shape[0]), dims="chain")).values,
            "draw":  posterior.get("draw",  xr.DataArray(np.arange(yhat.shape[1]), dims="draw")).values,
            "obs":   np.arange(n_obs),
        },
        name="structural_preds",
    )

# Residuals at posterior mean (for scatter)
median_pred = structural_preds.mean(("chain","draw")).values
median_res  = y_obs - median_pred          # should be ~centered

# ---------- Grab the same beta_orig ArviZ is plotting ----------
beta_da = idata2.posterior["beta_orig"]

# If beta has a covariate dimension, pick the same one ArviZ uses
if "covariates" in beta_da.dims:
    if "avexpr" in beta_da.coords["covariates"].values:
        beta_da = beta_da.sel(covariates="avexpr")
    else:
        beta_da = beta_da.isel(covariates=0)

# Flatten chains/draws and drop NaNs/Infs
beta_samples = np.ravel(beta_da.values).astype(float)
beta_samples = beta_samples[np.isfinite(beta_samples)]

# Use HDI (to match az.plot_posterior)
hdi_lo, hdi_hi = az.hdi(beta_samples, hdi_prob=0.95)
beta_med = float(np.median(beta_samples))

# ---------- Make the figure (β-based version) ----------
fig, ax2 = plt.subplots(figsize=(8, 6))

# Density plot (hist ok; ArviZ uses KDE, but HDI numbers will match now)
ax2.hist(beta_samples, bins=50, density=True, alpha=0.75, edgecolor="black")

# Annotations that now match ArviZ's HDI
ax2.axvline(beta_med, color="red", linewidth=2.5, label=f"Median: {beta_med:.3f}")
ax2.axvline(hdi_lo,  color="blue", linestyle="--", linewidth=2.0)
ax2.axvline(hdi_hi,  color="blue", linestyle="--", linewidth=2.0)
ax2.legend([plt.Line2D([], [], color='red', lw=2.5),
            plt.Line2D([], [], color='blue', lw=2.0, ls='--')],
           [f"Median: {beta_med:.3f}",
            f"95% HDI: [{hdi_lo:.3f}, {hdi_hi:.3f}]"])

ax2.axvline(0.0, color="black", linestyle=":", linewidth=2.0, label="Zero")
ax2.set_xlabel("β (protection against expropriation)")
ax2.set_ylabel("Posterior Density")
ax2.set_title("Posterior Distribution of β")

plt.tight_layout()
plt.show()

print(f"β median: {beta_med:.3f} | 95% HDI: [{hdi_lo:.3f}, {hdi_hi:.3f}] | Contains 0: {hdi_lo <= 0 <= hdi_hi}")


#### Simulations over delta-space

In [ ]:
# ---- 1) Choose a grid of *fixed* δ values (violations of exclusion)
delta_grid = np.linspace(-0.5, 0.0, 8)   

# ---- 2) Decide covariates 
data_path = r"C:\Users\B375471\Downloads\Acemoglu_osv\colonial_origins\maketable4\maketable4.dta"
df_ajr, stats = ajr.load_ajr_data(data_path, baseline_only=True, standardize=True)
covariates = []  # or None

# Helper to build priors that FIX δ at delta0 
def priors_fix_delta(delta0, covariates=None):
    # infer lengths so the prior vectors match 
    p = len(covariates) if covariates else 0
    FLAT = 1                        

    
    # First-stage: [intercept, covs..., logem4]
    mus_t = [0] + [0]*p + [0]         
    sig_t = [FLAT] + [FLAT]*p + [FLAT] 
    
    # Structural: [intercept, avexpr, covs..., logem4(delta)]
    mus_z = [0] + [0]*p + [0] + [delta0] # delta (logem4 coeff) gets mean delta0
    sig_z = [FLAT] + [FLAT]*p +[FLAT] + [delta0/1.96] # delta gets tiny sd (fixed)] 

    return {
        "mus":    [mus_t, mus_z],
        "sigmas": [sig_t, sig_z],
        "eta":    2,      # LKJ on residual corr 
        "lkj_sd": 2,      # Scale on sigmas 
    }

# ---- 3) Run the grid and collect summaries


fits, labels, rows = [], [], []
for i, d0 in enumerate(tqdm(delta_grid, desc="Delta grid simulation")):
    print(f"\n--- Running δ = {d0:+.2f} ({i+1}/{len(delta_grid)}) ---")
    
    pri = priors_fix_delta(d0, covariates=covariates)
    out = f"ajr_iv_fixdelta_{d0:+.2f}.nc"

    # Debug: Print prior structure
    print(f"First-stage equation mus: {pri['mus'][0]}")  # beta_z is structural
    print(f"Structural equation mus: {pri['mus'][1]}")  # beta_t is first-stage
    print(f"Expected: delta in first-stage = {d0} with tiny sd")

    iv, idata = ajr.run(
        data = df_ajr,
        loosen_exclusion=True,
        covariates=None,
        priors=pri,                   # δ fixed at d0
        draws=2000, tune=1000, chains=4, cores=4,
        target_accept=0.95,
        netcdf_path=out,
        stats=stats,
        standardize=True,
    )

    fits.append(idata); labels.append(f"δ={d0:+.2f}")
    
    # Understanding the correct structure
    print("Beta_z coordinates (STRUCTURAL eq):", list(idata.posterior.coords['covariates'].values))
    print("Beta_t coordinates (FIRST-STAGE eq):", list(idata.posterior.coords['instruments'].values))
    
    covariates_list = list(idata.posterior.coords['covariates'].values)
    instruments_list = list(idata.posterior.coords['instruments'].values)
    
    # Find avexpr coefficient in STRUCTURAL equation (beta_z)
    try:
        avexpr_idx = covariates_list.index('avexpr')
    except ValueError:
        # If not found by name, assume it's at index 1 (after intercept)
        avexpr_idx = 1
        print(f"Warning: 'avexpr' not found in covariates, using index 1")
    
    # Find logem4 coefficient in FIRST-STAGE equation (beta_t) 
    try:
        logem4_idx = instruments_list.index('logem4')
    except ValueError:
        # If not found by name, use last index
        logem4_idx = -1
        print(f"Warning: 'logem4' not found in instruments, using last index")
    
    # Extract coefficients from CORRECT equations
    avexpr_coeff = idata.posterior["beta_z"].isel(covariates=avexpr_idx)  # STRUCTURAL: avexpr effect on logpgp95
    delta_coeff = idata.posterior["beta_z"].isel(covariates=logem4_idx)   # FIRST-STAGE: delta (logem4 effect on avexpr)
    
    avexpr_mean = float(avexpr_coeff.mean())
    delta_mean = float(delta_coeff.mean())
    
    # FIXED: HDI for avexpr coefficient using numpy percentiles as fallback
    try:
        avexpr_hdi = az.hdi(avexpr_coeff)
        print(f"HDI type: {type(avexpr_hdi)}")
        print(f"HDI structure: {avexpr_hdi}")
        
        # Try different extraction methods
        if hasattr(avexpr_hdi, 'data_vars'):
            # It's a Dataset
            var_name = list(avexpr_hdi.data_vars.keys())[0]
            hdi_array = avexpr_hdi[var_name].values
            hdi_low = float(hdi_array[0])
            hdi_high = float(hdi_array[1])
        else:
            # It's a DataArray
            hdi_array = avexpr_hdi.values
            hdi_low = float(hdi_array[0])
            hdi_high = float(hdi_array[1])
            
    except Exception as e:
        print(f"HDI extraction failed: {e}")
        # Fallback to manual percentiles
        avexpr_samples = avexpr_coeff.values.flatten()
        hdi_low = float(np.percentile(avexpr_samples, 2.5))
        hdi_high = float(np.percentile(avexpr_samples, 97.5))
    
    rows.append({
        "delta_fixed": d0,
        "delta_actual_mean": delta_mean,  # Should be close to d0
        "avexpr_causal_effect": avexpr_mean,  # This is the causal effect of institutions
        "avexpr_hdi_low": hdi_low,
        "avexpr_hdi_high": hdi_high,
        "delta_sd_used": 1e-6,
        "file": out
    })
    
    print(f"✓ δ fixed at {d0:+.2f}, actual mean: {delta_mean:+.3f}")
    print(f"  Avexpr causal effect: {avexpr_mean:.3f}")
    print(f"  HDI: [{hdi_low:.3f}, {hdi_high:.3f}]")
    print(f"  Available covariates (beta_z): {covariates_list}")
    print(f"  Available instruments (beta_t): {instruments_list}")
    print(f"  Used avexpr_idx={avexpr_idx}, logem4_idx={logem4_idx}")

summary = pd.DataFrame(rows).sort_values("delta_fixed")
print("\n=== SUMMARY TABLE ===")
summary

In [ ]:


# ---- 1) The same grid used to run the model ----------------------------------
delta_grid = np.linspace(-0.5, 0.0, 8) 
def nc_name(d): 
    return f"ajr_iv_fixdelta_{d:+.2f}.nc"  

rows = []

for d0 in delta_grid:
    fname = nc_name(d0)
    if not os.path.exists(fname):
        print(f"WARNING: missing file {fname} — skipping")
        continue

    # Load inference data
    idata = az.from_netcdf(fname)

    # Get coordinate labels
    covars = list(idata.posterior.coords.get("covariates", []).values)
    instrs = list(idata.posterior.coords.get("instruments", []).values)

    # Find indices safely
    # Structural eq: beta_z over "covariates" (contains 'avexpr' and 'logem4')
    try:
        avexpr_idx = covars.index("avexpr")
    except Exception:
        avexpr_idx = 1  # fallback: intercept at 0, avexpr next
        print(f"NOTE: 'avexpr' not found in covariates; using index {avexpr_idx}")

    try:
        logem4_idx = covars.index("logem4")  # δ lives in STRUCTURAL coeffs here
    except Exception:
        logem4_idx = len(covars) - 1  # fallback: assume last
        print(f"NOTE: 'logem4' not found in covariates; using last index {logem4_idx}")

    # Extract draws
    post = idata.posterior
    avexpr_draws = post["beta_orig"].values.ravel()
    delta_draws  = post["delta_orig"].values.ravel()

    # Summary stats
    avexpr_mean = float(np.mean(avexpr_draws))
    # Try HDI first; if it fails, use equal-tail percentiles
    try:
        hdi = az.hdi(avexpr_draws, hdi_prob=0.95)
        # az.hdi may return shape (2,), make sure to order low<high
        lo, hi = float(np.min(hdi)), float(np.max(hdi))
    except Exception:
        lo, hi = float(np.percentile(avexpr_draws, 2.5)), float(np.percentile(avexpr_draws, 97.5))

    delta_mean = float(np.mean(delta_draws))

    rows.append({
        "delta_fixed": float(d0),
        "delta_actual_mean": delta_mean,
        "avexpr_causal_effect": avexpr_mean,
        "avexpr_hdi_low": lo,
        "avexpr_hdi_high": hi,
        "delta_sd_used": 1e-6,   # as in your prior
        "file": fname
    })

# ---- 2) Create DataFrame -----------------------------------------------------
df = pd.DataFrame(rows).sort_values("delta_fixed").reset_index(drop=True)
print(df)

# ---- 3) Plot the delta-sensitivity curve ------------------------------------
x  = df["delta_fixed"].to_numpy()
y  = df["avexpr_causal_effect"].to_numpy()
lo = df["avexpr_hdi_low"].to_numpy()
hi = df["avexpr_hdi_high"].to_numpy()

plt.figure(figsize=(7.5, 4.5))
plt.fill_between(x, lo, hi, alpha=0.2, label="95% HDI")
plt.plot(x, y, marker="o", linewidth=2, label="Posterior mean β ($A_i$ → log$Y_i$)")

# --------------------------------------------------------------------------
# 1) Highlight δ = -0.24
# --------------------------------------------------------------------------
delta_highlight = -0.24
beta_highlight = np.interp(delta_highlight, x, y)

# Vertical + horizontal reference lines
plt.plot([delta_highlight, delta_highlight], [0, beta_highlight],
         color="black", linestyle="--", linewidth=1.5, label=r"$\delta=-0.24$")
plt.plot([x.min(), delta_highlight], [beta_highlight, beta_highlight],
         color="black", linestyle=":", linewidth=1.2)
plt.scatter(delta_highlight, beta_highlight, color="black", zorder=5)



# --------------------------------------------------------------------------
# 2) Highlight β = 0.38
# --------------------------------------------------------------------------
beta_ref = 0.38
# Interpolate corresponding δ value
delta_ref = np.interp(beta_ref, y, x)

# Horizontal + vertical reference lines
plt.plot([x.min(), delta_ref], [beta_ref, beta_ref],
         color="purple", linestyle="--", linewidth=1.5, label=r"$\beta=0.38$")
plt.plot([delta_ref, delta_ref], [0, beta_ref],
         color="purple", linestyle=":", linewidth=1.2)
plt.scatter(delta_ref, beta_ref, color="purple", zorder=5)



# --------------------------------------------------------------------------
# Formatting
# --------------------------------------------------------------------------
plt.axvline(0, linestyle="--", linewidth=1)
plt.axhline(0, linestyle=":", linewidth=1)
plt.title("δ-sensitivity curve (β vs. fixed δ)")
plt.xlabel("Fixed δ (direct effect of log settler mortality)")
plt.ylabel("Posterior mean β on institutions")
plt.legend(loc="best")
plt.xlim(-0.5, None)
plt.savefig("delta_sensitivity_curve_highlighted.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# ---- Priors (aligned to standardized design) ----
# instruments (beta_t dims): [Intercept, logem4_std]
# structural (beta_z dims):  [Intercept, avexpr_std, logem4_std]  (only if loosen_exclusion=True)
priors_delta = {
    "mus": [
        [0.0, 0.0],                 # beta_t means: Intercept, logem4_std
        [0.0, 0.0, -0.2946],         # beta_z means: Intercept, avexpr_std, logem4_std
    ],
    "sigmas": [
        [1.0, 1.0],                 # beta_t sds
        [1.0, 1.0, 1.0],           # beta_z sds (tight prior on logem4_std if loosened)
    ],
    "eta": 2,       # LKJ shape (correlations); η=2 = mild shrinkage toward 0A
    "lkj_sd": 2,    # rate for Exponential prior on residual SDs (if you kept Exponential)
    "S": 0.46,      # Sensitivity parameter S (larger = wider delta prior)
}

data_path = r"C:\Users\B375471\Downloads\Acemoglu_osv\colonial_origins\maketable4\maketable4.dta"

# New loader returns (df, stats); we only need df here (run() standardizes internally)
df_ajr, stats = ajr.load_ajr_data(data_path, baseline_only=True, standardize=True)

# ---- Fit model ----
iv_obj, idata = ajr.run(
    data=df_ajr,              
    priors=priors_delta,
    draws=2000,
    tune=1000,
    chains=4,
    cores=4,
    target_accept=0.9,
    loosen_exclusion=True,    
    covariates=None,          
    netcdf_path="ajr_iv_delta_posterior07.nc",
    standardize=True, 
    stats=stats        
)


In [ ]:
idata3 = az.from_netcdf("ajr_iv_delta_posterior07.nc")
az.plot_trace(idata3, var_names=["beta_z", "beta_t"])
az.plot_posterior(idata3, var_names=["beta_orig", "delta_orig", "intercept_orig"], hdi_prob=0.95)


In [ ]:
az.plot_trace(
    idata3,
    var_names=["beta_z", "beta_t"],
    labeller=az.labels.MapLabeller(
        var_name_map={"beta_z": "Structural parameters", "beta_t": "First stage parameters"}
    )
)



idata3_plot = idata3.copy()

# Find which dimension indexes beta_z (often "beta_z_dim_0" or similar)
dim_name = idata3_plot.posterior["beta_z"].dims[-1]

# Replace the coordinate label
idata3_plot.posterior = idata3_plot.posterior.assign_coords({
    dim_name: [
        "Intercept" if x == "Intercept" else
        "Protection against expropriation" if x == "avexpr" else
        "Settler mortality" if x == "logem4" else x
        for x in idata3_plot.posterior[dim_name].values
    ]
})

coef_dim = idata3_plot.posterior["beta_z"].dims[-1]
idata_no_intercept = idata3_plot.copy()
idata_no_intercept.posterior = idata_no_intercept.posterior.drop_sel({coef_dim: "Intercept"})

az.plot_posterior(
    idata_no_intercept,
    var_names=["beta_z"],
    hdi_prob=0.95,
    labeller=az.labels.MapLabeller(var_name_map={"beta_z": "Structural parameters"})
)

In [ ]:


axes = az.plot_posterior(
    idata3,
    var_names=["beta_orig", "delta_orig"],
    hdi_prob=0.95,
    figsize=(10, 4),
    labeller=az.labels.MapLabeller(
        var_name_map={
            "beta_orig": r"$\beta$ (Protection against expropriation)",
            "delta_orig": r"$\delta$ (Settler mortality)"
        }
    )
)

# Extract the figure from one of the axes
fig = axes[0].get_figure()

# Add a shared title
fig.suptitle("Structural Parameters", fontsize=14, y=1.05)

plt.tight_layout()
plt.show()

In [ ]:
posterior = idata3.posterior
y_obs = df_ajr["logpgp95"].to_numpy()
z_inst = df_ajr["logem4"].to_numpy()

# ---------- Build structural predictions: yhat(chain, draw, obs) ----------
def find_obs_dim(da):
    for cand in ("obs", "observation", "i", "n", "N", "obs_id"):
        if cand in da.dims:
            return cand
    return None

beta_z = posterior["beta_z"]
obs_dim = find_obs_dim(beta_z)

if obs_dim is not None:
    structural_preds = beta_z  # already (chain, draw, obs)
else:
    if "covariates" not in beta_z.dims:
        raise ValueError(f"'beta_z' needs obs dim or 'covariates'. Got {beta_z.dims}.")
    covs_all = [str(c) for c in beta_z.coords["covariates"].values]
    intercept_names = {"Intercept", "intercept", "const", "Constant", "_cons"}
    use_explicit_intercept = "intercept_orig" in posterior.data_vars
    covs = [c for c in covs_all if not (use_explicit_intercept and c in intercept_names)]

    n_obs = len(df_ajr)
    cols = []
    for c in covs:
        if c in intercept_names:
            cols.append(np.ones((n_obs, 1)))  # only if no explicit intercept param
        else:
            if c not in df_ajr.columns:
                raise KeyError(f"Missing covariate in df_ajr: {c!r}")
            cols.append(df_ajr[c].to_numpy().reshape(-1, 1))
    X = np.hstack(cols) if cols else np.zeros((n_obs, 0))

    B = beta_z.sel(covariates=covs).values                      # (chain, draw, K)
    yhat = np.einsum("cdk,nk->cdn", B, X) if X.shape[1] else np.zeros(B.shape[:2] + (n_obs,))
    if use_explicit_intercept:
        yhat = yhat + posterior["intercept_orig"].values[..., None]

    structural_preds = xr.DataArray(
        yhat, dims=("chain","draw","obs"),
        coords={
            "chain": posterior.get("chain", xr.DataArray(np.arange(yhat.shape[0]), dims="chain")).values,
            "draw":  posterior.get("draw",  xr.DataArray(np.arange(yhat.shape[1]), dims="draw")).values,
            "obs":   np.arange(n_obs),
        },
        name="structural_preds",
    )

# Residuals at posterior mean (for scatter)
median_pred = structural_preds.mean(("chain","draw")).values
median_res  = y_obs - median_pred          # should be ~centered

# ---------- Grab the same beta_orig ArviZ is plotting ----------
beta_da = idata3.posterior["beta_orig"]

# If beta has a covariate dimension, pick the same one ArviZ uses
if "covariates" in beta_da.dims:
    if "avexpr" in beta_da.coords["covariates"].values:
        beta_da = beta_da.sel(covariates="avexpr")
    else:
        beta_da = beta_da.isel(covariates=0)

# Flatten chains/draws and drop NaNs/Infs
beta_samples = np.ravel(beta_da.values).astype(float)
beta_samples = beta_samples[np.isfinite(beta_samples)]

# Use HDI (to match az.plot_posterior)
hdi_lo, hdi_hi = az.hdi(beta_samples, hdi_prob=0.95)
beta_med = float(np.median(beta_samples))

# ---------- Make the figure (β-based version) ----------
fig, ax2 = plt.subplots(figsize=(8, 6))

# Density plot (hist ok; ArviZ uses KDE, but HDI numbers will match now)
ax2.hist(beta_samples, bins=50, density=True, alpha=0.75, edgecolor="black")

# Annotations that now match ArviZ's HDI
ax2.axvline(beta_med, color="red", linewidth=2.5, label=f"Median: {beta_med:.3f}")
ax2.axvline(hdi_lo,  color="blue", linestyle="--", linewidth=2.0)
ax2.axvline(hdi_hi,  color="blue", linestyle="--", linewidth=2.0)
ax2.legend([plt.Line2D([], [], color='red', lw=2.5),
            plt.Line2D([], [], color='blue', lw=2.0, ls='--')],
           [f"Median: {beta_med:.3f}",
            f"95% HDI: [{hdi_lo:.3f}, {hdi_hi:.3f}]"])

ax2.axvline(0.0, color="black", linestyle=":", linewidth=2.0, label="Zero")
ax2.set_xlabel("β (protection against expropriation)")
ax2.set_ylabel("Posterior Density")
ax2.set_title("Posterior Distribution of β")

plt.tight_layout()
plt.show()

print(f"β median: {beta_med:.3f} | 95% HDI: [{hdi_lo:.3f}, {hdi_hi:.3f}] | Contains 0: {hdi_lo <= 0 <= hdi_hi}")

In [ ]:
# Flatten posterior samples
beta = idata3.posterior["beta_orig"].values.flatten()
delta = idata3.posterior["delta_orig"].values.flatten()

# ---- Aesthetic setup ----
az.style.use("seaborn-v0_8-darkgrid")
plt.figure(figsize=(7, 6))

# ---- KDE contour ----
ax = sns.kdeplot(
    x=delta,
    y=beta,
    fill=True,
    levels=[0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95],
    cmap="mako",         # or "mako", "crest", "rocket", "cividis"
    linewidths=0.5
)

# ---- Labels and title ----
ax.set_xlabel(r"$\delta$ (Settler mortality)", fontsize=13, labelpad=10)
ax.set_ylabel(r"$\beta$ (Protection against expropriation)", fontsize=13, labelpad=10)
ax.set_title(r"Joint Posterior of $\beta$ and $\delta$", fontsize=15, pad=15)

# ---- Reference lines ----
ax.axvline(0, color="black", ls="--", lw=1)
ax.axhline(0, color="black", ls="--", lw=1)

# ---- Margins, ticks, and style ----
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", which="major", labelsize=11)
ax.grid(alpha=0.2)

# ---- Optional: mean point marker ----
ax.scatter(delta.mean(), beta.mean(), color="red", s=40, marker="x", label="Posterior mean")
ax.legend(frameon=False, loc="upper right", fontsize=11)
ax.set_xlim(-1.0, 0.4)
ax.set_ylim(-0.8, 1.5)


plt.show()


In [ ]:

# Grid of S hyperparameters
S_grid = [0.0001, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# Helper to build priors
def priors_with_S(S_val, covariates=None):
    p = len(covariates) if covariates else 0
    FLAT = 1.0
    mus_t = [0.0] + [0.0]*p + [0.0]
    sig_t = [FLAT] + [FLAT]*p + [FLAT]
    mus_z = [0.0] + [0.0]*p + [0.0] + [0.0]  # delta mean = 0
    sig_z = [FLAT] + [FLAT]*p + [FLAT] + [FLAT]
    return {
        "mus": [mus_t, mus_z],
        "sigmas": [sig_t, sig_z],
        "eta": 2,
        "lkj_sd": 2,
        "S": S_val,
    }

rows = []
fits = []

for S_val in S_grid:
    print(f"\n--- Running model with S = {S_val} ---")
    pri = priors_with_S(S_val)

    out_name = f"ajr_iv_condS_{S_val:.2f}.nc"
    iv, idata = ajr.run(
        data=df_ajr,
        priors=pri,
        loosen_exclusion=True,
        covariates=None,
        draws=2000,
        tune=1000,
        chains=4,
        cores=4,
        target_accept=0.9,
        netcdf_path=out_name,
        stats=stats,
        standardize=True,
    )

    fits.append(idata)

    beta_mean = float(idata.posterior["beta_orig"].mean())
    beta_draws = idata.posterior["beta_orig"].values.ravel()
    # Use equal-tail 95% as a simple, stable fallback
    beta_low, beta_high = np.quantile(beta_draws, [0.025, 0.975])
    beta_low, beta_high = float(beta_low), float(beta_high)


    rows.append({
        "S": S_val,
        "beta_mean": beta_mean,
        "beta_hdi_low": beta_low,
        "beta_hdi_high": beta_high,
    })

summary = pd.DataFrame(rows)
print(summary)


In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.fill_between(summary["S"], summary["beta_hdi_low"], summary["beta_hdi_high"],
                 alpha=0.2, label="95% posterior interval")
plt.plot(summary["S"], summary["beta_mean"], marker="o", linewidth=2,
         label=r"Posterior mean $\beta$ (Institutions → log GDP)")


plt.axhline(0, linestyle=":", color="k", lw=1)

plt.xlabel(r"Prior scale $\kappa$ in $\delta \mid \beta \sim N(0,(\kappa\beta)^2)$")
plt.ylabel(r"Posterior mean of $\beta$")
plt.title(r"Sensitivity of $\beta$ to prior scale $\kappa$")
plt.legend()
plt.tight_layout()
plt.savefig("beta_sensitivity_to_S.png", dpi=160)
plt.show()


In [ ]:


# Use the same filtered data from above
first_stage_preds = idata.posterior['mu'].isel(mu_dim_1=0)  # First stage predictions
observed_avexpr = df_ajr['avexpr'].values

print(f"Shape check - first_stage_preds: {first_stage_preds.shape}, observed_avexpr: {observed_avexpr.shape}")

# CORRECT instrument relevance check: correlation between instrument (logem4) and treatment (avexpr)
instrument_treatment_corr = np.corrcoef(df_ajr['logem4'], df_ajr['avexpr'])[0,1]
print(f"Instrument relevance (logem4 vs avexpr): {instrument_treatment_corr:.3f}")
print("Rule of thumb: Should be > 0.3 for reasonable instrument strength")

# First-stage model fit: how well does the model predict avexpr?
if first_stage_preds.shape[2] == len(observed_avexpr):
    # Add first-stage PPC for visualization
    idata.posterior_predictive = idata.posterior_predictive.assign(
        avexpr=first_stage_preds.rename(mu_dim_0='avexpr_dim_0')
    )
    observed_avexpr_da = xr.DataArray(
        observed_avexpr, 
        dims=['avexpr_dim_0'],
        coords={'avexpr_dim_0': range(len(observed_avexpr))}
    )
    idata.observed_data = idata.observed_data.assign(avexpr=observed_avexpr_da)

    # Plot first-stage fit
    az.plot_ppc(idata, data_pairs={"avexpr": "avexpr"}, var_names=["avexpr"], figsize=(10, 6))
    
    # Model fit metrics (R-squared equivalent)
    pred_mean = first_stage_preds.mean(['chain', 'draw'])
    ss_res = np.sum((observed_avexpr - pred_mean)**2)
    ss_tot = np.sum((observed_avexpr - np.mean(observed_avexpr))**2)
    r_squared = 1 - (ss_res / ss_tot)
    print(f"First-stage R-squared: {r_squared:.3f}")
    
    
else:
    print("ERROR: Dimension mismatch!")
    print(f"Model predictions: {first_stage_preds.shape[2]}, Data observations: {len(observed_avexpr)}")



In [ ]:

idata3 = az.from_netcdf("ajr_iv_delta_posterior07.nc")

# Extract structural equation predictions (logpgp95) — UNCHANGED
structural_preds = idata3.posterior['mu'].isel(mu_dim_1=1)  # (chain, draw, obs)

observed_logpgp95 = df_ajr['logpgp95'].values

# --- center ONLY the observed ---
obs_mean = observed_logpgp95.mean()
observed_logpgp95_centered = observed_logpgp95 - obs_mean

# Add the logpgp95 predictions to posterior_predictive (UNSHIFTED)
idata3.posterior_predictive = idata3.posterior_predictive.assign(
    logpgp95=structural_preds.rename(mu_dim_0='logpgp95_dim_0')
)

# Add the CENTERED observed logpgp95 to observed_data (same var name)
observed_da = xr.DataArray(
    observed_logpgp95_centered,
    dims=['logpgp95_dim_0'],
    coords={'logpgp95_dim_0': range(len(observed_logpgp95_centered))}
)
idata3.observed_data = idata3.observed_data.assign(logpgp95=observed_da)

# Remove likelihood if present
if 'likelihood' in idata3.posterior_predictive:
    idata3.posterior_predictive = idata3.posterior_predictive.drop_vars('likelihood')
if 'likelihood' in idata3.observed_data:
    idata3.observed_data = idata3.observed_data.drop_vars('likelihood')

# PPC: predictive "logpgp95" vs observed "logpgp95" (now centered)
az.plot_ppc(
    idata3,
    data_pairs={"logpgp95": "logpgp95"},
    var_names=["logpgp95"],
    figsize=(10, 6),
    kind="kde",                 # smoother
    num_pp_samples=200,         # fewer overlays
    random_seed=1,
)

# Set the ylabel after creating the plot
plt.ylabel("Centered $Y_i$ (log GDP per capita)")
plt.show()

print(f"Observed mean (raw): {obs_mean:.3f}")
print(f"Observed mean (centered): {observed_logpgp95_centered.mean():.3f}")
print(f"Predicted mean (unchanged): {float(structural_preds.mean()):.3f}")
print(f"Observed std (raw): {observed_logpgp95.std():.3f}")
print(f"Predicted std: {float(structural_preds.std()):.3f}")
